# Assistants 

[Assistants](https://docs.langchain.com/langsmith/assistants) give developers a quick and easy way to modify and version agents for experimentation.

## Supplying configuration to the graph

Our `task_maistro` graph is already set up to use assistants!

It has a `configuration.py` file defined and loaded in the graph.

We access configurable fields (`user_id`, `todo_category`, `task_maistro_role`) inside the graph nodes.

## Creating assistants 

Now, what is a practical use case for assistants with the `task_maistro` app that we've been building?

For me, it's the ability to have separate ToDo lists for different categories of tasks. 

For example, I want one assistant for my personal tasks and another for my work tasks.

These are easily configurable using the `todo_category` and `task_maistro_role` configurable fields.

![Screenshot 2024-11-18 at 9.35.55 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/673d50597f4e9eae9abf4869_Screenshot%202024-11-19%20at%206.57.01%E2%80%AFPM.png)

This is the default assistant that we created when we deployed the graph.

In [1]:
from langgraph_sdk import get_client
url_for_cli_deployment = "http://localhost:8123"
client = get_client(url=url_for_cli_deployment)

### Personal assistant

This is the personal assistant that I'll use to manage my personal tasks.

In [8]:
personal_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    graph_id="task_maistro", 
    config={"configurable": {"todo_category": "personal"}}
    ,name="Personal Assistant",
    description="A personal assistant to help me manage my personal tasks"
)
print(personal_assistant)

{'assistant_id': '722e86eb-15b5-476b-b37a-48147982ac84', 'graph_id': 'task_maistro', 'version': 1, 'created_at': '2026-04-04T06:14:36.436707+00:00', 'updated_at': '2026-04-04T06:14:36.436707+00:00', 'config': {'configurable': {'todo_category': 'personal'}}, 'context': {'todo_category': 'personal'}, 'metadata': {}, 'name': 'Personal Assistant', 'description': 'A personal assistant to help me manage my personal tasks'}


Let's update this assistant to include my `user_id` for convenience,  [creating a new version of it](https://docs.langchain.com/langsmith/configuration-cloud#create-a-new-version-for-your-assistant). 

In [9]:
task_maistro_role = """You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:

- Help track and organize personal tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- Proactively ask for deadlines when new tasks are added without them
- Maintain a supportive tone while helping the user stay accountable
- Help prioritize tasks based on deadlines and importance

Your communication style should be encouraging and helpful, never judgmental. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Would you like to add one to help us track it better?"""

configurations = {"todo_category": "personal", 
                  "user_id": "Mario",
                  "task_maistro_role": task_maistro_role
                  }

personal_assistant = await client.assistants.update(
    personal_assistant["assistant_id"],
    config={"configurable": configurations}
)
print(personal_assistant)

{'assistant_id': '722e86eb-15b5-476b-b37a-48147982ac84', 'graph_id': 'task_maistro', 'version': 2, 'created_at': '2026-04-04T06:14:36.436707+00:00', 'updated_at': '2026-04-04T06:14:50.255000+00:00', 'config': {'configurable': {'task_maistro_role': 'You are a friendly and organized personal task assistant. Your main focus is helping users stay on top of their personal tasks and commitments. Specifically:\n\n- Help track and organize personal tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- Proactively ask for deadlines when new tasks are added without them\n- Maintain a supportive tone while helping the user stay accountable\n- Help prioritize tasks based on deadlines and importance\n\nYour communication style should be encouraging and helpful, never judgmental. \n\

### Work assistant

Now, let's create a work assistant. I'll use this for my work tasks.

In [13]:
task_maistro_role = """You are a focused and efficient work task assistant. 

Your main focus is helping users manage their work commitments with realistic timeframes. 

Specifically:

- Help track and organize work tasks
- When providing a 'todo summary':
  1. List all current tasks grouped by deadline (overdue, today, this week, future)
  2. Highlight any tasks missing deadlines and gently encourage adding them
  3. Note any tasks that seem important but lack time estimates
- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:
  • Developer Relations features: typically 1 day
  • Course lesson reviews/feedback: typically 2 days
  • Documentation sprints: typically 3 days
- Help prioritize tasks based on deadlines and team dependencies
- Maintain a professional tone while helping the user stay accountable

Your communication style should be supportive but practical. 

When tasks are missing deadlines, respond with something like "I notice [task] doesn't have a deadline yet. Based on similar tasks, this might take [suggested timeframe]. Would you like to set a deadline with this in mind?"""

configurations = {"todo_category": "work", 
                  "user_id": "Mario",
                  "task_maistro_role": task_maistro_role
                  }

work_assistant = await client.assistants.create(
    # "task_maistro" is the name of a graph we deployed
    graph_id="task_maistro",
    config={"configurable": configurations}
    ,name= "Work Assistant"
    ,description= "A work assistant to help me manage my work tasks with realistic timeframes"
)
print(work_assistant)

{'assistant_id': '9bb3d24b-16f9-440b-bf75-2265667c1a87', 'graph_id': 'task_maistro', 'version': 1, 'created_at': '2026-04-04T06:16:59.820163+00:00', 'updated_at': '2026-04-04T06:16:59.820163+00:00', 'config': {'configurable': {'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 days\n- Help prioritize tasks base

## Using assistants 

Assistants will be saved to `Postgres` in our deployment.  

This allows us to easily search <!--[~search~](https://langchain-ai.github.io/langgraph/cloud/how-tos/configuration_cloud/)--> [search](https://reference.langchain.com/python/langsmith/deployment/sdk/#langgraph_sdk.client.AssistantsClient.search) for assistants with the SDK.

In [14]:
assistants = await client.assistants.search()
for assistant in assistants:
    print({
        'assistant_id': assistant['assistant_id'],
        'version': assistant['version'],
        'config': assistant['config']
    })

{'assistant_id': '9bb3d24b-16f9-440b-bf75-2265667c1a87', 'version': 1, 'config': {'configurable': {'task_maistro_role': 'You are a focused and efficient work task assistant. \n\nYour main focus is helping users manage their work commitments with realistic timeframes. \n\nSpecifically:\n\n- Help track and organize work tasks\n- When providing a \'todo summary\':\n  1. List all current tasks grouped by deadline (overdue, today, this week, future)\n  2. Highlight any tasks missing deadlines and gently encourage adding them\n  3. Note any tasks that seem important but lack time estimates\n- When discussing new tasks, suggest that the user provide realistic time-frames based on task type:\n  • Developer Relations features: typically 1 day\n  • Course lesson reviews/feedback: typically 2 days\n  • Documentation sprints: typically 3 days\n- Help prioritize tasks based on deadlines and team dependencies\n- Maintain a professional tone while helping the user stay accountable\n\nYour communicati

We can manage them easily with the SDK. For example, we can delete assistants that we're no longer using.  
> The syntax in the video is slightly off. The updated code below creates a spare assistant and then deletes it. 

In [15]:
# create a temporary assitant
temp_assistant = await client.assistants.create(
    "task_maistro", 
    config={"configurable": configurations}
)

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"before delete: {{'assistant_id': {assistant['assistant_id']}}}")
    
# delete our temporary assistant
await client.assistants.delete(assistants[-1]["assistant_id"])
print()

assistants = await client.assistants.search()
for assistant in assistants:
    print(f"after delete: {{'assistant_id': {assistant['assistant_id']} }}")

before delete: {'assistant_id': 8fb9dc15-1a11-4f2c-a72d-0c9c9bab811e}
before delete: {'assistant_id': 9bb3d24b-16f9-440b-bf75-2265667c1a87}
before delete: {'assistant_id': 793909d8-323b-4142-a4cf-31a095f4ff08}
before delete: {'assistant_id': 722e86eb-15b5-476b-b37a-48147982ac84}
before delete: {'assistant_id': c9419032-fb31-4081-9a5b-a8f913f5d2ba}
before delete: {'assistant_id': 9f1848f0-9aa8-4c11-97a4-082d7f89721c}
before delete: {'assistant_id': 1e4d7195-1f9a-49d9-b864-243347dfa082}
before delete: {'assistant_id': ea4ebafa-a81d-5063-a5fa-67c755d98a21}

after delete: {'assistant_id': 8fb9dc15-1a11-4f2c-a72d-0c9c9bab811e }
after delete: {'assistant_id': 9bb3d24b-16f9-440b-bf75-2265667c1a87 }
after delete: {'assistant_id': 793909d8-323b-4142-a4cf-31a095f4ff08 }
after delete: {'assistant_id': 722e86eb-15b5-476b-b37a-48147982ac84 }
after delete: {'assistant_id': c9419032-fb31-4081-9a5b-a8f913f5d2ba }
after delete: {'assistant_id': 9f1848f0-9aa8-4c11-97a4-082d7f89721c }
after delete: {'ass

Let's set the assistant IDs for the `personal` and `work` assistants that I'll work with.

In [16]:
work_assistant_id = assistants[0]['assistant_id']
personal_assistant_id = assistants[1]['assistant_id']

### Work assistant

Let's add some ToDos for my work assistant.

In [17]:
from langchain_core.messages import HumanMessage
from langchain_core.messages import convert_to_messages

user_input = "crea o actualiza algunos ToDos: 1) Re-film Module 6, lesson 5 para el final del día de hoy. 2) Update audioUX para el próximo lunes."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

crea o actualiza algunos ToDos: 1) Re-film Module 6, lesson 5 para el final del día de hoy. 2) Update audioUX para el próximo lunes.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_Gm23RDcGK2Nn1NFP8SNrYNsB)
 Call ID: call_Gm23RDcGK2Nn1NFP8SNrYNsB
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': 'Re-film Module 6, lesson 5', 'time_to_complete': 120, 'deadline': '2026-04-04T23:59:00', 'status': 'not started'}
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_7WjFHuHiLNDSnyXYJ0nRYnDj)
 Call ID: call_7WjFHuHiLNDSnyXYJ0nRYnDj
  Args:
    update_type: todo
================================= Tool Message =================================

Document e76b8e03-8f3a-46fb-88cd-9ccc020fc50a unch

In [18]:
user_input = "Necesito hacer un resumen de los tutorials que he seguido hasta ahora."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Necesito hacer un resumen de los tutorials que he seguido hasta ahora.
================================== Ai Message ==================================

Para ayudarte a organizar esta tarea, sería útil saber si tienes una fecha límite en mente para completar el resumen de los tutoriales. Además, ¿cuánto tiempo crees que te llevará completar esta tarea? Esto nos ayudará a priorizar y gestionar tus tareas de manera más efectiva.


The assistant uses it's instructions to push back with task creation! 

It asks me to specify a deadline :) 

In [19]:
user_input = "Esto debe tardar unos 3 dias y me gustaria terminarlo antes del mes"
async for chunk in client.runs.stream(thread["thread_id"], 
                                      work_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Esto debe tardar unos 3 dias y me gustaria terminarlo antes del mes
================================== Ai Message ==================================

Perfecto, entonces podemos establecer una fecha límite para el resumen de los tutoriales para el 31 de marzo. Voy a añadir esta tarea a tu lista con un tiempo estimado de 3 días para completarla. Un momento mientras actualizo tu lista de tareas.
Tool Calls:
  UpdateMemory (call_ZIHUDp5GGIT6Hnb0BHyK90yc)
 Call ID: call_ZIHUDp5GGIT6Hnb0BHyK90yc
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': 'Hacer un resumen de los tutoriales seguidos', 'time_to_complete': 4320, 'deadline': '2026-04-30T23:59:00', 'status': 'not started'}

Document 09c162f0-0937-4188-9b9e-0040c57d0c1c unchanged:
No changes needed for this task.

Document e76b8e03-8f3a-46fb-88cd-9ccc020fc50a unch

### Personal assistant

Similarly, we can add ToDos for my personal assistant.

In [21]:
user_input = "Quehaceres: comprar huevos de pascua, organizar fiesta de cumpleaños para mi hijo el próximo mes, reservar cita médica para el próximo lunes."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Quehaceres: comprar huevos de pascua, organizar fiesta de cumpleaños para mi hijo el próximo mes, reservar cita médica para el próximo lunes.
================================== Ai Message ==================================
Tool Calls:
  UpdateMemory (call_eYad6BG18FyaOaSG5SA9b3Zr)
 Call ID: call_eYad6BG18FyaOaSG5SA9b3Zr
  Args:
    update_type: todo
================================= Tool Message =================================

New ToDo created:
Content: {'task': 'Comprar huevos de pascua', 'time_to_complete': 60, 'deadline': '2026-04-09T23:59:00', 'solutions': ['Visitar la tienda de dulces local', 'Buscar ofertas en línea', 'Comparar precios en supermercados'], 'status': 'not started'}

New ToDo created:
Content: {'task': 'Organizar fiesta de cumpleaños para mi hijo', 'time_to_complete': 240, 'deadline': '2026-05-15T23:59:00', 'solutions': ['Elegir un tema para la fiesta', 'Hacer una lista de invitados

In [22]:
user_input = "Dame mi resumen de tareas pendientes y vencidas para esta semana."
thread = await client.threads.create()
async for chunk in client.runs.stream(thread["thread_id"], 
                                      personal_assistant_id,
                                      input={"messages": [HumanMessage(content=user_input)]},
                                      stream_mode="values"):

    if chunk.event == 'values':
        state = chunk.data
        convert_to_messages(state["messages"])[-1].pretty_print()

================================ Human Message =================================

Dame mi resumen de tareas pendientes y vencidas para esta semana.
================================== Ai Message ==================================

Aquí tienes el resumen de tus tareas pendientes y vencidas para esta semana:

### Tareas Vencidas
- **Re-film Module 6, lesson 5**: Debía completarse el 4 de abril de 2026.

### Tareas para Hoy (6 de abril de 2026)
- **Reservar cita médica**: Debe completarse hoy.
- **Update audioUX**: Debe completarse hoy.

### Tareas para Esta Semana
- **Comprar huevos de pascua**: Debe completarse antes del 9 de abril de 2026.

### Tareas Futuras
- **Hacer un resumen de los tutoriales seguidos**: Debe completarse antes del 29 de abril de 2026.
- **Organizar fiesta de cumpleaños para mi hijo**: Debe completarse antes del 15 de mayo de 2026.

### Tareas sin Plazo
- **Update audioUX**: No tiene un tiempo estimado para completar. ¿Te gustaría establecer un tiempo estimado para 